In [3]:
%pip install youtube_comment_downloader 

Note: you may need to restart the kernel to use updated packages.


In [4]:
from youtube_comment_downloader import YoutubeCommentDownloader
import pandas as pd
import time
import os
from urllib.parse import urlparse, parse_qs

# -----------------------------
# CONFIGURAÇÃO
# -----------------------------

TEMA = "Novas regras CNH"

links = [
    "https://www.youtube.com/watch?v=AyRf35aU5eg",
    "https://www.youtube.com/watch?v=UZpv03WjizA",
    "https://www.youtube.com/watch?v=ECEuQGBhd0c",
    "https://www.youtube.com/watch?v=vpQ98fwomAU"
]

MAX_COMENTARIOS_POR_VIDEO = 500

# -----------------------------
# FUNÇÃO EXTRAIR VIDEO ID
# -----------------------------

def extrair_video_id(url):
    """Extrai o ID do vídeo de forma segura usando urlparse"""
    parsed_url = urlparse(url)
    # Pega o valor do parâmetro 'v' na query string da URL
    return parse_qs(parsed_url.query).get('v', [None])[0]

# -----------------------------
# SCRAPER
# -----------------------------

downloader = YoutubeCommentDownloader()
dados = []

for url in links:
    print(f"\nColetando comentários de: {url}")
    
    contador = 0
    video_id = extrair_video_id(url)
    
    if not video_id:
        print(f"Não foi possível extrair o ID do vídeo da URL: {url}")
        continue

    try:
        for comentario in downloader.get_comments_from_url(url):
            dados.append({
                "tema": TEMA,
                "video_id": video_id,
                "comentario": comentario["text"],
                "likes": comentario.get("votes", 0),
                "autor": comentario.get("author", "Anônimo"),
                "fonte": url
            })
            
            contador += 1
            
            if contador >= MAX_COMENTARIOS_POR_VIDEO:
                break
                
        print(f"{contador} comentários coletados")
        time.sleep(2)
        
    except Exception as e:
        print(f"Erro ao coletar {url}: {e}")

# -----------------------------
# DATAFRAME
# -----------------------------

df = pd.DataFrame(dados)

if not df.empty:
    print("\nAmostra dos dados:")
    print(df.head())

# -----------------------------
# EXPORTAÇÃO
# -----------------------------

# Cria a pasta 'Dados raspados' no mesmo local onde o script está rodando, se ela não existir
diretorio = "Dados raspados"
os.makedirs(diretorio, exist_ok=True)

arquivo_saida = os.path.join(diretorio, "dataset_opinioes_cnh_youtube.csv")

if not df.empty:
    df.to_csv(
        arquivo_saida,
        index=False,
        encoding="utf-8-sig"
    )
    print(f"\nArquivo salvo em: {arquivo_saida}")
    print(f"Total de registros: {len(df)}")
else:
    print("\nNenhum dado foi coletado para salvar.")


Coletando comentários de: https://www.youtube.com/watch?v=AyRf35aU5eg
Erro ao coletar https://www.youtube.com/watch?v=AyRf35aU5eg: Failed to set sorting

Coletando comentários de: https://www.youtube.com/watch?v=UZpv03WjizA
500 comentários coletados

Coletando comentários de: https://www.youtube.com/watch?v=ECEuQGBhd0c
238 comentários coletados

Coletando comentários de: https://www.youtube.com/watch?v=vpQ98fwomAU
189 comentários coletados

Amostra dos dados:
               tema     video_id  \
0  Novas regras CNH  UZpv03WjizA   
1  Novas regras CNH  UZpv03WjizA   
2  Novas regras CNH  UZpv03WjizA   
3  Novas regras CNH  UZpv03WjizA   
4  Novas regras CNH  UZpv03WjizA   

                                          comentario likes  \
0  Maravilhoso menos custos para ambos agora prec...     0   
1                                           Kkkkkkkl     0   
2                                     Glória há Deus     0   
3  Parabéns ao presidente Lula, ajudando o povo b...     0   
4  Mais 